# VisionTrack — YOLO26 Training Walkthrough

This notebook walks through fine-tuning YOLO26 on a custom dataset.

**Sections:**
1. Dataset Preparation
2. Training Configuration
3. Training Execution
4. Evaluation & Metrics
5. ONNX Export
6. Inference with Custom Model

## 1. Dataset Preparation

YOLO training requires a dataset YAML file pointing to your images and labels.

### Directory Structure
```
data/custom/
├── train/
│   ├── images/    # Training images (.jpg, .png)
│   └── labels/    # YOLO-format labels (.txt)
├── val/
│   ├── images/    # Validation images
│   └── labels/    # Validation labels
└── custom.yaml    # Dataset config
```

### Label Format (YOLO)
Each image has a corresponding `.txt` file with one line per object:
```
<class_id> <x_center> <y_center> <width> <height>
```
All values are normalized to [0, 1] relative to image dimensions.

In [ ]:
import sys
sys.path.insert(0, '../python')

from utils.dataset import create_dataset_yaml, validate_dataset_yaml

# Example: Create a dataset YAML for a custom 3-class detection task
# Uncomment and modify for your own dataset:

# create_dataset_yaml(
#     dataset_dir='../data/custom',
#     train_path='../data/custom/train/images',
#     val_path='../data/custom/val/images',
#     class_names=['cat', 'dog', 'bird'],
#     output_yaml='../data/custom/custom.yaml'
# )

## 2. Training Configuration

Key hyperparameters for YOLO26 training:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `epochs` | 100 | Number of training epochs |
| `imgsz` | 640 | Input image size |
| `batch` | 16 | Batch size |
| `lr0` | 0.01 | Initial learning rate |
| `patience` | 50 | Early stopping patience |
| `model` | yolo26n | Model variant (n/s/m/l/x) |

In [ ]:
# Training parameters (modify as needed)
TRAIN_ARGS = {
    'model': 'yolo26n',        # Model variant
    'data': 'custom.yaml',     # Dataset YAML (relative to data/)
    'epochs': 100,             # Training epochs
    'imgsz': 640,              # Input size
    'batch': 16,               # Batch size
    'lr0': 0.01,               # Initial learning rate
    'patience': 50,            # Early stopping
    'device': 'cpu',           # 'cpu', '0', or '0,1' for multi-GPU
}

print('Training configuration:')
for k, v in TRAIN_ARGS.items():
    print(f'  {k}: {v}')

## 3. Training Execution

Run the training script. This will:
1. Download YOLO26 pretrained weights (first run only)
2. Start training with your dataset
3. Save checkpoints to `runs/detect/train/`
4. Run validation at the end

In [ ]:
# Option A: Run training as a subprocess
# import subprocess
# cmd = [
#     'python', '../python/train.py',
#     '--model', TRAIN_ARGS['model'],
#     '--data', TRAIN_ARGS['data'],
#     '--epochs', str(TRAIN_ARGS['epochs']),
#     '--imgsz', str(TRAIN_ARGS['imgsz']),
#     '--batch', str(TRAIN_ARGS['batch']),
#     '--lr', str(TRAIN_ARGS['lr0']),
# ]
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print('ERROR:', result.stderr)

In [ ]:
# Option B: Use Ultralytics API directly
# from ultralytics import YOLO
#
# model = YOLO('yolo26n.pt')  # Load pretrained
# results = model.train(
#     data='custom.yaml',
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     lr0=0.01,
#     patience=50,
#     project='runs/detect',
#     name='custom_train',
# )
# print(f'Training complete! Results saved to runs/detect/custom_train/')

## 4. Evaluation & Metrics

After training, evaluate on the validation set:

- **mAP@50**: Mean Average Precision at IoU=0.5
- **mAP@50:95**: Mean Average Precision at IoU=0.5:0.95
- **Precision**: True positives / (True positives + False positives)
- **Recall**: True positives / (True positives + False negatives)

In [ ]:
# Load trained model and run validation
# from ultralytics import YOLO
#
# model = YOLO('runs/detect/custom_train/weights/best.pt')
# metrics = model.val(data='custom.yaml')
#
# print(f'mAP@50:    {metrics.box.map50:.3f}')
# print(f'mAP@50-95: {metrics.box.map:.3f}')
# print(f'Precision:  {metrics.box.mp:.3f}')
# print(f'Recall:     {metrics.box.mr:.3f}')

## 5. ONNX Export

Export the trained model to ONNX format for deployment:

```bash
python python/export_onnx.py --weights runs/detect/custom_train/weights/best.pt --simplify
```

In [ ]:
# Export to ONNX
# from export_onnx import export, validate_export
#
# onnx_path = export(
#     weights='runs/detect/custom_train/weights/best.pt',
#     imgsz=640,
#     simplify=True
# )
# validate_export(onnx_path)

## 6. Inference with Custom Model

Run inference with your fine-tuned model:

```bash
# Python
python python/demo_tracker.py --model models/best.onnx --source image.jpg

# C++
./build/visiontrack detect image.jpg --model models/best.onnx

# Gradio
python python/demo_gradio.py --model models/best.onnx
```